In [1]:
import json
from pathlib import Path
import re
from collections import defaultdict

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import pandas as pd

nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

In [2]:
corpus = {
    "D1": "Information retrieval helps users find relevant documents.",
    "D2": "Information Retrieval systems index documents and documents.",
    "D3": "Search engines retrieve relevant web documents.",
    "D4": "Document indexing supports efficient information retrieval.",
    "D5": "Machine learning improves search engine ranking.",
    "D6": "Search systems use indexed documents to retrieve information.",
    "D7": "The search engine retrieves documents; the documents are searchable.",
    "D8": "Indexing documents makes information retrieval more efficient.",
    "D9": "DOCUMENTS contain information about document indexing.",
    "D10": "Retrieval systems organise documents and document collections.",
}

In [3]:
def preprocess_document(text: str) -> list[str]:
    """Version A : minuscules, suppression de la ponctuation et tokenisation."""
    text_lower = text.lower()
    text_no_punct = re.sub(r"[^a-z0-9\s]", " ", text_lower)
    return text_no_punct.split()


def filter_stopwords(tokens: list[str], stopword_set: set[str]) -> list[str]:
    """Version B : retire les stop words de la liste de tokens."""
    return [token for token in tokens if token not in stopword_set]


def apply_stemming(tokens: list[str], porter: PorterStemmer) -> list[str]:
    """Version C : applique le PorterStemmer à chaque token."""
    return [porter.stem(token) for token in tokens]


tokens_A = {doc_id: preprocess_document(text) for doc_id, text in corpus.items()}
tokens_B = {doc_id: filter_stopwords(toks, stop_words) for doc_id, toks in tokens_A.items()}
tokens_C = {doc_id: apply_stemming(toks, stemmer) for doc_id, toks in tokens_B.items()}

for doc_id in corpus:
    print(f"=== {doc_id} ===")
    print("  A:", tokens_A[doc_id])
    print("  B:", tokens_B[doc_id])
    print("  C:", tokens_C[doc_id])

=== D1 ===
  A: ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
  B: ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
  C: ['inform', 'retriev', 'help', 'user', 'find', 'relev', 'document']
=== D2 ===
  A: ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']
  B: ['information', 'retrieval', 'systems', 'index', 'documents', 'documents']
  C: ['inform', 'retriev', 'system', 'index', 'document', 'document']
=== D3 ===
  A: ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
  B: ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
  C: ['search', 'engin', 'retriev', 'relev', 'web', 'document']
=== D4 ===
  A: ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
  B: ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval']
  C: ['document', 'index', 'support', 'effici', 'inform', 'retriev']
=== D5 ===
  A: ['machine', 'learnin

In [4]:
def build_vocabulary(tokenized_docs: dict[str, list[str]]) -> list[str]:
    """Retourne la liste triée des termes uniques d'une collection."""
    vocab = set()
    for tokens in tokenized_docs.values():
        vocab.update(tokens)
    return sorted(vocab)


def build_term_document_matrix(
    tokenized_docs: dict[str, list[str]], vocab: list[str]
) -> dict[str, dict[str, int]]:
    """Matrice binaire : 1 si le terme est présent dans le document, sinon 0."""
    doc_ids = list(tokenized_docs.keys())
    doc_sets = {doc_id: set(tokens) for doc_id, tokens in tokenized_docs.items()}
    return {
        term: {doc_id: (1 if term in doc_sets[doc_id] else 0) for doc_id in doc_ids}
        for term in vocab
    }


def build_tf_matrix(
    tokenized_docs: dict[str, list[str]], vocab: list[str]
) -> dict[str, dict[str, int]]:
    """Matrice Term Frequency (TF) : nombre d'occurrences du terme par document."""
    doc_ids = list(tokenized_docs.keys())
    return {
        term: {doc_id: tokenized_docs[doc_id].count(term) for doc_id in doc_ids}
        for term in vocab
    }


vocab_A = build_vocabulary(tokens_A)
vocab_B = build_vocabulary(tokens_B)
vocab_C = build_vocabulary(tokens_C)

bin_A = build_term_document_matrix(tokens_A, vocab_A)
tf_A = build_tf_matrix(tokens_A, vocab_A)

bin_B = build_term_document_matrix(tokens_B, vocab_B)
tf_B = build_tf_matrix(tokens_B, vocab_B)

bin_C = build_term_document_matrix(tokens_C, vocab_C)
tf_C = build_tf_matrix(tokens_C, vocab_C)

print("Vocabulaire A (taille =", len(vocab_A), "):", vocab_A)
print("\nMatrice binaire A :")
display(pd.DataFrame.from_dict(bin_A, orient="index"))

print("\nMatrice TF A :")
display(pd.DataFrame.from_dict(tf_A, orient="index"))

Vocabulaire A (taille = 36 ): ['about', 'and', 'are', 'collections', 'contain', 'document', 'documents', 'efficient', 'engine', 'engines', 'find', 'helps', 'improves', 'index', 'indexed', 'indexing', 'information', 'learning', 'machine', 'makes', 'more', 'organise', 'ranking', 'relevant', 'retrieval', 'retrieve', 'retrieves', 'search', 'searchable', 'supports', 'systems', 'the', 'to', 'use', 'users', 'web']

Matrice binaire A :


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
about,0,0,0,0,0,0,0,0,1,0
and,0,1,0,0,0,0,0,0,0,1
are,0,0,0,0,0,0,1,0,0,0
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,1,1,0,0,1,1,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0



Matrice TF A :


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
about,0,0,0,0,0,0,0,0,1,0
and,0,1,0,0,0,0,0,0,0,1
are,0,0,0,0,0,0,1,0,0,0
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,2,1,0,0,1,2,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0


In [5]:
# 3.1 Index manuel sur D1-D3 (version A, positions à partir de 0)
# D1: ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents']
# D2: ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents']
# D3: ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents']
manual_index_D1_D3 = {
    "information": {"D1": [0], "D2": [0]},
    "retrieval": {"D1": [1], "D2": [1]},
    "documents": {"D1": [6], "D2": [4, 6], "D3": [5]},
}


def build_inverted_index(
    tokenized_docs: dict[str, list[str]],
) -> dict[str, dict[str, list[int]]]:
    """Construit un index inversé positionnel : terme -> {doc_id: [positions_0_basees]}."""
    temp_index: dict[str, dict[str, list[int]]] = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in tokenized_docs.items():
        for pos, token in enumerate(tokens):
            temp_index[token][doc_id].append(pos)

    return {
        term: {doc_id: list(positions) for doc_id, positions in docs.items()}
        for term, docs in sorted(temp_index.items())
    }


def search_term(
    inverted_index: dict[str, dict[str, list[int]]], term: str
) -> list[str]:
    """Retourne la liste des identifiants de documents contenant le terme."""
    if term not in inverted_index:
        return []
    return list(inverted_index[term].keys())


inv_index_A = build_inverted_index(tokens_A)
inv_index_B = build_inverted_index(tokens_B)
inv_index_C = build_inverted_index(tokens_C)

print("Extrait index A ('documents') :", inv_index_A.get("documents"))
print("Recherche 'retrieval' en version A :", search_term(inv_index_A, "retrieval"))

Extrait index A ('documents') : {'D1': [6], 'D2': [4, 6], 'D3': [5], 'D6': [4], 'D7': [4, 6], 'D8': [1], 'D9': [0], 'D10': [3]}
Recherche 'retrieval' en version A : ['D1', 'D2', 'D4', 'D8', 'D10']


In [6]:
def summarize_version(
    version: str,
    tokenized_docs: dict[str, list[str]],
    inv_index: dict[str, dict[str, list[int]]],
) -> dict:
    """Calcule la taille du vocabulaire, la moyenne de tokens/doc, le total de postings et de positions."""
    vocab = build_vocabulary(tokenized_docs)
    total_tokens = sum(len(toks) for toks in tokenized_docs.values())
    avg_tokens = total_tokens / len(tokenized_docs)
    total_postings = sum(len(docs) for docs in inv_index.values())
    total_positions = sum(
        len(positions) for docs in inv_index.values() for positions in docs.values()
    )
    return {
        "Version": version,
        "Vocabulary size": len(vocab),
        "Average tokens per document": round(avg_tokens, 2),
        "Total postings": total_postings,
        "Total stored positions": total_positions,
    }


comparison_df = pd.DataFrame([
    summarize_version("A", tokens_A, inv_index_A),
    summarize_version("B", tokens_B, inv_index_B),
    summarize_version("C", tokens_C, inv_index_C),
])

display(comparison_df)

,Version,Vocabulary size,Average tokens per document,Total postings,Total stored positions
0,A,36,6.9,66,69
1,B,30,6.1,59,61
2,C,24,6.1,57,61


In [7]:
# 1 & 2. Tests de search_term sur 4 cas :
# - Un seul document
single_doc_res = search_term(inv_index_A, "helps")
assert single_doc_res == ["D1"], f"Attendu ['D1'], obtenu {single_doc_res}"

# - Plusieurs documents
multi_doc_res = search_term(inv_index_A, "information")
assert multi_doc_res == ["D1", "D2", "D4", "D6", "D8", "D9"], (
    f"Résultat inattendu pour 'information' : {multi_doc_res}"
)

# - Terme dont la forme change après stemming (comparaison B et C)
assert "indexing" in inv_index_B and "indexing" not in inv_index_C
assert "index" in inv_index_C
assert search_term(inv_index_C, "index") == ["D2", "D4", "D6", "D8", "D9"]

# - Terme inconnu
unknown_res = search_term(inv_index_A, "unknownword")
assert unknown_res == [], f"Attendu [], obtenu {unknown_res}"

# 3. Vérification des positions dans l'index inversé pour chaque version
for ver_name, toks_dict, idx in [
    ("A", tokens_A, inv_index_A),
    ("B", tokens_B, inv_index_B),
    ("C", tokens_C, inv_index_C),
]:
    for term, docs in idx.items():
        for doc_id, positions in docs.items():
            for pos in positions:
                assert 0 <= pos < len(toks_dict[doc_id]), f"Position hors bornes : {pos}"
                assert toks_dict[doc_id][pos] == term, (
                    f"Écart de position en {ver_name} pour '{term}' dans {doc_id} à pos={pos}"
                )

# 4. Comparaison avec l'index manuel sur D1-D3 (version A)
for term, manual_docs in manual_index_D1_D3.items():
    auto_docs_d1_d3 = {
        doc_id: pos_list
        for doc_id, pos_list in inv_index_A[term].items()
        if doc_id in {"D1", "D2", "D3"}
    }
    assert auto_docs_d1_d3 == manual_docs, (
        f"Écart manuel/auto sur D1-D3 pour '{term}' : {auto_docs_d1_d3} != {manual_docs}"
    )

# 5. Vérification des matrices binaire et TF
for ver_name, toks_dict, vocab, b_mat, tf_m in [
    ("A", tokens_A, vocab_A, bin_A, tf_A),
    ("B", tokens_B, vocab_B, bin_B, tf_B),
    ("C", tokens_C, vocab_C, bin_C, tf_C),
]:
    for term in vocab:
        for doc_id, toks in toks_dict.items():
            expected_tf = toks.count(term)
            expected_bin = 1 if expected_tf > 0 else 0
            assert b_mat[term][doc_id] in (0, 1)
            assert b_mat[term][doc_id] == expected_bin
            assert tf_m[term][doc_id] == expected_tf

print("Toutes les vérifications de la Partie 5 sont validées !")

Toutes les vérifications de la Partie 5 sont validées !


In [8]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

# 1. Tokens des versions A, B et C
for ver, data in [("A", tokens_A), ("B", tokens_B), ("C", tokens_C)]:
    with open(results_dir / f"tokens_{ver}.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

# 2. Vocabulaires des versions A, B et C
for ver, vocab in [("A", vocab_A), ("B", vocab_B), ("C", vocab_C)]:
    with open(results_dir / f"vocabulary_{ver}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(vocab) + "\n")

# 3. Matrices binaires et TF des versions A, B et C
for ver, toks_dict, vocab, b_mat, tf_m in [
    ("A", tokens_A, vocab_A, bin_A, tf_A),
    ("B", tokens_B, vocab_B, bin_B, tf_B),
    ("C", tokens_C, vocab_C, bin_C, tf_C),
]:
    pd.DataFrame.from_dict(b_mat, orient="index").to_csv(
        results_dir / f"binary_matrix_{ver}.csv", index_label="term"
    )
    pd.DataFrame.from_dict(tf_m, orient="index").to_csv(
        results_dir / f"tf matrix {ver}.csv", index_label="term"
    )

# 4. Index inversés des versions A, B et C
for ver, idx in [("A", inv_index_A), ("B", inv_index_B), ("C", inv_index_C)]:
    with open(results_dir / f"inverted_index_{ver}.json", "w", encoding="utf-8") as f:
        json.dump(idx, f, indent=2, ensure_ascii=False)

# 5. Tableau comparatif du prétraitement
comparison_df.to_csv(results_dir / "preprocessing_comparison.csv", index=False)

print("Tous les fichiers de results/ ont été générés :", sorted(p.name for p in results_dir.iterdir()))

Tous les fichiers de results/ ont été générés : ['binary_matrix_A.csv', 'binary_matrix_B.csv', 'binary_matrix_C.csv', 'inverted_index_A.json', 'inverted_index_B.json', 'inverted_index_C.json', 'preprocessing_comparison.csv', 'tf matrix A.csv', 'tf matrix B.csv', 'tf matrix C.csv', 'tokens_A.json', 'tokens_B.json', 'tokens_C.json', 'vocabulary_A.txt', 'vocabulary_B.txt', 'vocabulary_C.txt']


In [9]:
%%writefile requirements.txt
nltk>=3.8
pandas>=2.0

Writing requirements.txt


In [10]:
%%writefile README.md
# IR-TP1-Document-Preprocessing-and-Indexing

## How to Run the Notebook
1. Open `tpl_preprocessing_indexing.ipynb` in Google Colab or Jupyter Notebook.
2. Run all cells sequentially (`Runtime -> Run all` in Colab).
3. Inspect the generated outputs inside the `results/` directory.

## Implementation Choices
- **Preprocessing**: Lowercasing, regex punctuation removal (`[^a-z0-9\s]`), whitespace tokenization, NLTK English stop-word filtering, and NLTK `PorterStemmer` applied sequentially across versions A, B, and C.
- **Representations**: Sorted vocabularies, binary term-document matrices, and term-frequency (TF) matrices computed directly in Python and formatted via `pandas` for display and CSV export.
- **Positional Inverted Index**: Built using `collections.defaultdict` mapping each term to document IDs and zero-based token positions, queried through `search_term()`.

## Main Results & Interpretation
Lowercasing merges case variants such as capitalized titles or uppercase terms, while removing stop words eliminates frequent function words, reducing vocabulary size, total postings, and stored token positions while shifting positions of remaining tokens. Applying Porter stemming further decreases vocabulary size and postings by grouping inflected or morphological variants under common stems. However, preprocessing does not always improve retrieval: dropping stop words can alter exact phrase or specific queries, and stemming may over-stem distinct concepts or under-stem valid variants, potentially affecting precision or recall depending on the query.

Writing README.md
